In [1]:
import numpy as np 
import pandas as pd 

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(train.shape, test.shape)
train.head()

# Check for Any missing values?
print("Missing values:\n", train.isna().sum().sum())

def clean(df):
    df = df.copy()
    # fold undocumented codes into the "other" category
    df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
    df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})
    return df

# clean up data that has unidenfitied values
train = clean(train)
test = clean(test)

train_copy = train.copy()
print(train_copy["default"].value_counts(normalize=True))

print(train_copy[["LIMIT_BAL", "AGE"]].describe())

print(train_copy.groupby("SEX")["default"].mean())
print(train_copy.groupby("EDUCATION")["default"].mean())
print(train_copy.groupby("MARRIAGE")["default"].mean())


# evaluate the delayed payment
repayment_columns = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
train_copy['num_period_late'] = (train_copy[repayment_columns] > 0).sum(axis=1)

# look at the average number of period for delayed payment
print(train_copy.groupby("num_period_late")["default"].mean())


# determine how the avg bill amount for non and default
bill_cols = [
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "BILL_AMT4", "BILL_AMT5", "BILL_AMT6"
]
train_copy["avg_bill"] = train_copy[bill_cols].mean(axis=1)
print(train_copy.groupby("default")["avg_bill"].mean())


# compare the avg payment amount per period for non and default
pay_amt_cols = ["PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"]
train_copy["avg_payment"] = train_copy[pay_amt_cols].mean(axis=1)
print(train_copy.groupby("default")["avg_payment"].mean())

# look at avg percentage of credit balance used across all BILL_AMOUNT 
train_copy["avg_credit_utilisation"] = (train_copy[bill_cols].mean(axis=1) / train_copy["LIMIT_BAL"])
print(train_copy.groupby("default")["avg_credit_utilisation"].mean())


#feautures engineering
def engineer_features(df):
    df = df.copy()

    repayment_columns = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
    bill_cols = ["BILL_AMT1", "BILL_AMT2", "BILL_AMT3","BILL_AMT4", "BILL_AMT5", "BILL_AMT6"]
    pay_amt_cols = ["PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"]

    df['num_period_late'] = (df[repayment_columns] > 0).sum(axis=1)
    df['max_period_late'] = df[repayment_columns].max(axis=1)
    df["avg_bill"] = df[bill_cols].mean(axis=1)
    df["avg_payment"] = df[pay_amt_cols].mean(axis=1)
    df["avg_credit_utilisation"] = (df[bill_cols].mean(axis=1) / df["LIMIT_BAL"])
    df["pay_to_bill_ratio"] = df[pay_amt_cols].sum(axis=1) / (df[bill_cols].sum(axis=1).abs() + 1)
    df['limit_per_age'] = df['LIMIT_BAL'] / df['AGE']
    df["bill_change"] = (df["BILL_AMT1"] - df["BILL_AMT6"])
    df["zero_payment_count"] = (df[pay_amt_cols] == 0).sum(axis=1)
    df["payment_std"] = (df[pay_amt_cols].std(axis=1))
    df["bill_std"] = df[bill_cols].std(axis=1)
    df["max_credit_utilisation"] = (df[bill_cols].max(axis=1)/ df["LIMIT_BAL"])

    delay_flags = (df[repayment_columns] > 0).astype(int)

    weights = np.array([6, 5, 4, 3, 2, 1])

    df["weighted_positive_delay"] = (delay_flags.values * weights).sum(axis=1)
    

    
    return df

train = engineer_features(train)
test = engineer_features(test) 

# Use CatBoost model to predict default probability
from catboost import CatBoostClassifier
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

# separate columns with categorical values not just numerical
categories = ['SEX', 'EDUCATION', 'MARRIAGE']

X = train.drop(columns=["default", "client_id"])
y = train["default"]
X_test = test.drop(columns=["client_id"])

# Create a stratified 5 fold ccross validation splitter to better average results
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=50
)

oof_probs = np.zeros(len(X)) # store the validation prediction for train
test_probs_cv = np.zeros(len(X_test))

# loop through each of the 5 fold
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    print(f"Fold {fold + 1}")

    # Split X into training and validation data for this fold
    X_train_fold = X.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]

    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=1500,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=5,
        loss_function="Logloss",
        eval_metric="Logloss",
        random_seed=50 + fold,
        verbose=False
    )

    model.fit(
        X_train_fold,
        y_train_fold,
        cat_features=categories,
        eval_set=(X_val_fold, y_val_fold),
        early_stopping_rounds=100
    )

    # validation predictions for this fold
    fold_val_probs = model.predict_proba(X_val_fold)[:, 1]
    oof_probs[val_idx] = fold_val_probs

    # test predictions from this fold
    test_probs_cv += model.predict_proba(X_test)[:, 1] / skf.n_splits

    # print("Fold log loss:", log_loss(y_val_fold, fold_val_probs))

print("Overall CV log loss:", log_loss(y, oof_probs))

# create submission csv
submission = pd.DataFrame({
    "client_id": test["client_id"],
    "default_probability": test_probs_cv
})

submission.to_csv(
    "submission_cv.csv",
    index=False
)

print(submission.head())


ModuleNotFoundError: No module named 'numpy'